In [ ]:
import os
import pandas as pd
import time
import re


BASE_DATA_PATH = r"/workspace/qgi\qgi-4.0\data"
EVENTS_PATH = os.path.join(BASE_DATA_PATH, "events")
SCDIS_PATH = os.path.join(BASE_DATA_PATH, "scdi_finder_output")
COUNTRIES_CSV_PATH = os.path.join(BASE_DATA_PATH, "countries.csv")
RECIPES_PATH = os.path.join(BASE_DATA_PATH, "recipes")
countries_df = pd.read_csv(COUNTRIES_CSV_PATH)

In [ ]:
def sanitize_filename(filename, lower = False):
    """Sanitizes a string to be used as a valid filename."""

    s = str(filename).strip().replace(" ", "_")
    s = re.sub(r"(?u)[^-\w.]", "", s)
    if not s:
        s = "_"
    if lower:
        return s.lower()
    return s

def load_events(events_path):
    """
    Loads events from text files in the specified directory.

    Args:
        events_path (str): Path to the directory containing event text files.

    Returns:
        dict: A dictionary where keys are country names and values are lists of event dictionaries.
    """
    events = {}
    if not os.path.exists(events_path):
        print(f"Warning: Events path not found: {events_path}")
        return events

    # Iterate through each file in the events directory
    for file in os.listdir(events_path):
        if file.endswith('.txt'):
            country_name = file.replace('.txt', '')
            file_path = os.path.join(events_path, file)
            try:
                # Read and parse the event file
                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()
            except Exception as e:
                print(f"Error reading event file {file_path}: {e}")
                continue

            # Process each line to extract event details
            country_events = []
            for line in lines:
                line = line.strip()
                if line.startswith('. '):
                    line = line[2:]  # Remove the leading '. '
                    parts = line.split(' | ')
                    if len(parts) == 4:
                        _discarded_country_name, year_str, event_type, description = parts
                        try:
                            country_events.append({
                                'year': int(year_str),
                                'event_type': event_type.strip(),
                                'description': description.strip()
                            })
                        except ValueError:
                            print(f"Warning: Could not parse year '{year_str}' in {file_path}")
            if country_events:
                events[country_name] = country_events
    return events

def get_name_by_id(input_id):
    """
    Searches for the specified country ID and returns the primary name (name_1).
    """
    if countries_df.empty:
        # print("Warning: Countries DataFrame is empty. Cannot get name by ID.")
        return None # Or handle as an error, or return input_id as string
    try:
        id_int = int(float(input_id)) # Handle cases like 2.0
        result = countries_df[countries_df["id"] == id_int]
        if not result.empty:
            return result["name_1"].iloc[0]
    except ValueError:
        # print(f"Warning: Invalid ID format for get_name_by_id: {input_id}")
        pass
    return None # Or return str(input_id) to have a fallback

def get_id_by_name(input_name):
    """
    Searches for the specified country name in columns name_1 through name_20
    (prioritizing name_1) and returns the corresponding country ID.

    Args:
        input_name (str): The country name to search for.

    Returns:
        int or None: The ID of the country if found, otherwise None.
    """
    if 'countries_df' not in globals() or countries_df.empty:
        print("Warning: Global 'countries_df' is not defined or is empty. Cannot get ID by name.")
        return None

    # Essential 'id' column check
    if "id" not in countries_df.columns:
        print("Warning: Column 'id' not found in countries_df. Cannot return ID.")
        return None

    if not isinstance(input_name, str) or not input_name.strip():
        # Input name must be a non-empty string
        return None

    normalized_input_name = input_name.lower().strip()

    # Define all name columns to check, in order of priority
    name_columns_to_search = ['name_1'] + [f'name_{i}' for i in range(2, 21)]

    for col_name in name_columns_to_search:
        if col_name in countries_df.columns:
            # Perform case-insensitive matching for the current column
            # .astype(str) handles potential NaNs by converting them to the string "nan",
            # which won't match a typical normalized country name.
            matches = countries_df[countries_df[col_name].astype(str).str.lower().str.strip() == normalized_input_name]
            if not matches.empty:
                return matches["id"].iloc[0]  # Return ID of the first match found

    return None # Return None if no match is found in any specified column

def get_fitting_events_for_scdi(scdi_row, events_data):
    """
    Returns event types that fit the given SCDI row.
    (Logic for end year remains as per original script: start_year + length)

    Args:
        scdi_row (pd.Series): A row from the SCDIs DataFrame.
        events_data (dict): A dictionary where keys are country names and values are lists of event dictionaries.

    Returns:
        list: A list of event types that fit the SCDI.
    """
    country_a_id     = scdi_row["country_a_id"]
    country_b_id     = scdi_row["country_b_id"]
    length           = scdi_row["length"]
    start_year_a_val = scdi_row["start_year_a"]
    start_year_b_val = scdi_row["start_year_b"]

    if any(v is None for v in [country_a_id, country_b_id, length, start_year_a_val, start_year_b_val]):
        # print("Warning: SCDI row is missing one or more required fields.")
        return []

    country_a_name = get_name_by_id(country_a_id)
    country_b_name = get_name_by_id(country_b_id)

    if not country_a_name or not country_b_name:
        # print(f"Warning: Could not find names for country IDs {country_a_id} or {country_b_id}")
        return []

    # Calculate end years for both countries
    # As per original logic and user confirmation to keep it for now:
    end_year_a = int(start_year_a_val + length) # TODO: -1 # add margin [-2, +2]
    end_year_b = int(start_year_b_val + length) # TODO: -1 # add margin [-2, +2]

    country_a_events = events_data.get(str(country_a_name), [])
    country_b_events = events_data.get(str(country_b_name), [])

    fitting_events_types = set()
    for event_a in country_a_events:
        if event_a["year"] == end_year_a:
            for event_b in country_b_events:
                if event_b["year"] == end_year_b and event_b["event_type"] == event_a["event_type"]:
                    fitting_events_types.add(event_a["event_type"])
    return list(fitting_events_types)

def save_or_merge_recipe_data(new_recipe_df, event_type_str, base_recipes_path, do_print_progress=True):
    """
    Saves a DataFrame to a Parquet file for a given event type.
    If the file already exists, it merges the new data with the existing data,
    removing duplicates based on indicator, country, and SCDI index.

    Args:
        new_recipe_df (pd.DataFrame): DataFrame containing new SCDI references for the event.
        event_type_str (str): The name of the event type.
        base_recipes_path (str): The base directory where recipe Parquet files are stored.
        do_print_progress (bool): Whether to print progress messages.
    """
    if new_recipe_df.empty:
        if do_print_progress:
            print(f"  No new data to save for event: {event_type_str}")
        return

    sanitized_name = sanitize_filename(event_type_str, lower = True) # Assumes sanitize_filename is defined elsewhere
    output_file_path = os.path.join(base_recipes_path, f"{sanitized_name}.parquet")

    # Define the unique key columns for an SCDI reference
    key_columns = ['indicator_name', 'country_name', 'scdi_row_index']

    # try:
    if os.path.exists(output_file_path):
        if do_print_progress:
            print(f"  Merging with existing recipe: \"\\{"\\".join(output_file_path.split("\\")[-2:])}\"...")
        existing_df = pd.read_parquet(output_file_path)
        
        # Ensure new_recipe_df has the same columns as existing_df, or handle schema evolution if necessary
        # For now, assuming schemas are compatible for concatenation
        combined_df = pd.concat([existing_df, new_recipe_df], ignore_index=True)
        
        # Remove duplicates based on the key columns
        # TODO: check logic here in the comment if it is what we want:
        # Keep the 'first' occurrence. If correlation might change/improve, 'last' might be better
        # to keep the newer data. Assuming 'first' is fine for now.
        rows_before_dedup = len(combined_df)
        final_df = combined_df.drop_duplicates(subset=key_columns, keep='first')
        rows_after_dedup = len(final_df)
        
        final_df.to_parquet(output_file_path, index=False)
        if do_print_progress:
            print(f"    Merged recipe saved:  \"\\{"\\".join(output_file_path.split("\\")[-2:])}\"...")
            print(f"    Total entries: {rows_after_dedup} (added {rows_after_dedup - len(existing_df)}, duplicates removed: {rows_before_dedup - rows_after_dedup}).")
    else:
        new_recipe_df.to_parquet(output_file_path, index=False)
        if do_print_progress:
            print(f"  New recipe saved: {output_file_path} with {len(new_recipe_df)} entries.")
    # except Exception as e:
    #     if do_print_progress:
    #         print(f"  Error processing recipe file {output_file_path}: {e}")

def browse_all_scdis_and_save_recipes(scdis_base_path, events_data, recipes_output_path, print_progress=True):
    """
    Processes all SCDIs, finds fitting events, and saves SCDI references to recipe Parquet files.
    """
    print("Starting to browse SCDIs and generate recipes...")
    all_event_recipes_data = {} # {event_type: [list_of_scdi_refs]}

    if not os.path.exists(scdis_base_path):
        print(f"Error: SCDIs base path not found: {scdis_base_path}")
        return

    indicator_folders = [d for d in os.listdir(scdis_base_path) if os.path.isdir(os.path.join(scdis_base_path, d))]
    total_indicator_folders = len(indicator_folders)
    if total_indicator_folders == 0:
        print(f"No indicator folders found in {scdis_base_path}")
        return

    for i, indicator_folder_name in enumerate(indicator_folders):
        indicator_path = os.path.join(scdis_base_path, indicator_folder_name)
        
        country_files = [f for f in os.listdir(indicator_path) if f.endswith(".parquet")]
        total_country_files = len(country_files)
        if total_country_files == 0:
            if print_progress: print(f"No Parquet files found in indicator folder: {indicator_folder_name}")
            continue

        for j, scdi_file_name in enumerate(country_files):
            parquet_country_name = scdi_file_name.replace(".parquet", "")
            file_path = os.path.join(indicator_path, scdi_file_name)

            try:
                scdis_df = pd.read_parquet(file_path)
            except Exception as e:
                if print_progress: print(f"Error reading Parquet file {file_path}: {e}")
                continue
            
            total_rows = len(scdis_df)
            if total_rows == 0:
                continue

            for k, scdi_row in scdis_df.iterrows():
                if print_progress:
                    indicator_percent = (i + 1) / total_indicator_folders * 100
                    country_percent = (j + 1) / total_country_files * 100
                    row_percent = (k + 1) / total_rows * 100
                    print(f"Processing: {indicator_folder_name} ({indicator_percent:.1f}%) > {parquet_country_name} ({country_percent:.1f}%) > Row {k} ({row_percent:.1f}%)")

                fitting_event_types = get_fitting_events_for_scdi(scdi_row, events_data)

                if fitting_event_types:
                    if print_progress:
                        row_country_a_name = get_name_by_id(scdi_row.get('country_a_id'))
                        row_country_b_name = get_name_by_id(scdi_row.get('country_b_id'))
                        print(f"  Found fitting events: {fitting_event_types} for SCDI ")
                        print(f"  (Countries: {row_country_a_name}-{row_country_b_name}, "
                              f"Index: {k} in {scdi_file_name})")
                    
                    for event_type in fitting_event_types:
                        scdi_reference = {
                            'indicator_name': indicator_folder_name,
                            'country_name': get_id_by_name(parquet_country_name), # Name of the parquet file (country)
                            'scdi_row_index': k, # Row index z
                            'correlation': scdi_row.get('correlation', None) # Make sure 'correlation' exists
                        }
                        if event_type not in all_event_recipes_data:
                            all_event_recipes_data[event_type] = []
                        all_event_recipes_data[event_type].append(scdi_reference)
                # break # one row
            # break # one parquet file
        break # one indicator folder
    
    print("Finished processing all SCDIs.")

    # Save accumulated recipes to Parquet files
    if not all_event_recipes_data:
        print("No recipes found/generated in this run.")
        # We still print "All recipe processing complete" at the end to signify the save phase is done.
    else:
        print(f"Saving/Merging recipes to \"\\{"\\".join(recipes_output_path.split("\\")[-2:])}\"...")
        for event_type, scdi_references_list in all_event_recipes_data.items():
            if not scdi_references_list: # Should not happen if event_type is in keys, but good check
                if print_progress: print(f"  No new references for event type '{event_type}' in this run.")
                continue

            # Create a DataFrame from the newly collected references for this event type for the current run
            current_run_recipe_df = pd.DataFrame(scdi_references_list)
            
            # Call the helper function to save or merge this DataFrame
            save_or_merge_recipe_data(current_run_recipe_df, event_type, recipes_output_path, print_progress)
            
    print("All recipe saving/merging operations complete.")

def analyze_recipe(event_name_input, recipes_data_path):
    """
    Reads a recipe Parquet file and prints an analysis of its contents.
    """
    print(f"\n--- Analyzing Recipe for Event: '{event_name_input}' ---")
    sanitized_event_name = sanitize_filename(event_name_input, lower = True)
    recipe_file_path = os.path.join(recipes_data_path, f"{sanitized_event_name}.parquet")

    if not os.path.exists(recipe_file_path):
        print(f"Error: Recipe file not found: {recipe_file_path}")
        return

    try:
        recipe_df = pd.read_parquet(recipe_file_path)
    except Exception as e:
        print(f"Error reading recipe file {recipe_file_path}: {e}")
        return

    if recipe_df.empty:
        print("Recipe is empty.")
        return

    print(f"Total SCDI references in this recipe: {len(recipe_df)}\n")

    # List indicators by frequency and average correlation
    if 'indicator_name' not in recipe_df.columns:
        print("Error: 'indicator_name' column not found in the recipe file.")
        return
    if 'correlation' not in recipe_df.columns:
        print("Warning: 'correlation' column not found in the recipe file. Cannot compute average correlation.")
        # List only by frequency
        indicator_counts = recipe_df['indicator_name'].value_counts().reset_index()
        indicator_counts.columns = ['Indicator Name', 'Frequency']
        print("Indicators by Frequency (Most to Least Frequent):")
        for _, row in indicator_counts.iterrows():
            print(f"  - {row['Indicator Name']}: {row['Frequency']} occurrences")

    else:
        # Calculate frequency
        indicator_counts = recipe_df['indicator_name'].value_counts()
        
        # Calculate average correlation
        # Ensure correlation is numeric, coerce errors to NaN, then dropna for mean calculation
        recipe_df['correlation_numeric'] = pd.to_numeric(recipe_df['correlation'], errors='coerce')
        avg_correlations = recipe_df.groupby('indicator_name')['correlation_numeric'].mean()
        
        print("Indicators by Frequency (Most to Least Frequent) with Average Correlation:")
        
        for indicator_name, count in indicator_counts.items():
            avg_corr = avg_correlations.get(indicator_name, float('nan')) # Get avg_corr, default to NaN
            print(f"  - Indicator: {indicator_name}")
            print(f"    Frequency: {count}")
            print(f"    Average Correlation: {avg_corr:.4f}" if not pd.isna(avg_corr) else "    Average Correlation: N/A")
            print("-" * 20)


loaded_events_data = load_events(EVENTS_PATH)
browse_all_scdis_and_save_recipes(SCDIS_PATH, loaded_events_data, RECIPES_PATH, print_progress=True)
# analyze_recipe("Judicial Reforms", RECIPES_PATH)


In [ ]:
analyze_recipe("Judicial Reforms", RECIPES_PATH)

### Starting Over

In [ ]:
import os
import pandas as pd
from IPython.display import clear_output
import time


class Recipe:
    """
    Represents a recipe for a specific event type, containing components that describe the indicators and sectors involved.

    Attributes:
        event_type (str): Type of event the recipe is for.
        components (list[tuple]): List of components in the recipe, each being a tuple of (index_name, sector, average_correlation, frequency, frequency_percentage).
    """

    def __init__(self, event_type):
        self.event_type = event_type
        self.components = (
            []
        )  # List of tuples (index_name, sector, average_correlation, frequency, frequency_percentage)

    def add_component(
        self, index_name, sector, average_correlation, frequency, frequency_percentage
    ):
        """
        Adds a component to the recipe.

        Args:
            index_name (str): Name of the index.
            sector (str): Sector associated with the index.
            average_correlation (float): Average correlation value.
            frequency (int): Frequency of the component.
            frequency_percentage (float): Frequency percentage.
        """
        self.components.append(
            (index_name, sector, average_correlation, frequency, frequency_percentage)
        )

    def to_dict(self):
        """
        Converts the recipe to a dictionary.

        Returns:
            dict: Dictionary representation of the recipe.
        """
        return {"event_type": self.event_type, "components": self.components}

    @classmethod
    def from_dict(cls, data):
        """
        Creates a Recipe object from a dictionary.

        Args:
            data (dict): Dictionary containing recipe data.

        Returns:
            Recipe: Recipe object created from the dictionary.
        """
        recipe = cls(data["event_type"])
        for component in data["components"]:
            recipe.add_component(*component)
        return recipe


def show_recipe(recipe: Recipe, print_out=True):
    """
    Displays a recipe's details in a readable format.

    Args:
        recipe (Recipe): The Recipe instance to display
        print_out (bool): If True, prints the recipe details. If False, returns them as a dictionary. Defaults to True.

    Returns:
        dict: Recipe details as a dictionary if print_out is False
    """
    if print_out:
        print(f"Event Type: {recipe.event_type}")
        print(f"Number of Components: {len(recipe.components)}")
        print("-" * 50)
        for component in recipe.components:
            index_name, sector, avg_corr, freq, freq_percent = component
            print(f"Index: {index_name}")
            print(f"Sector: {sector}")
            print(f"Average Correlation: {avg_corr:.2f}")
            print(f"Frequency: {freq} ({freq_percent:.1f}%)")
            print("-" * 50)

    return {
        "event_type": recipe.event_type,
        "components": [
            {
                "index_name": component[0],
                "sector": component[1],
                "average_correlation": component[2],
                "frequency": component[3],
                "frequency_percentage": component[4],
            }
            for component in recipe.components
        ],
    }


EVENTS_PATH = r"/workspace/qgi\qgi-4.0\data\events"
SCDIS_PATH = r"/workspace/qgi\qgi-4.0\data\scdi_finder_output"
countries_df = pd.read_csv(r"/workspace/qgi\qgi-4.0\data\countries.csv")

In [ ]:
def load_events(events_path):
    """
    Loads events from text files in the specified directory.

    Args:
        events_path (str): Path to the directory containing event text files.

    Returns:
        dict: A dictionary where keys are country names and values are lists of event dictionaries.
    """
    events = {}

    # Iterate through each file in the events directory
    for file in os.listdir(events_path):
        if file.endswith(".txt"):
            country_name = file.replace(".txt", "")
            file_path = os.path.join(events_path, file)

            # Read and parse the event file
            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Process each line to extract event details
            country_events = []
            for line in lines:
                line = line.strip()
                if line.startswith(". "):
                    line = line[2:]  # Remove the leading '. '
                    parts = line.split(" | ")
                    if len(parts) == 4:
                        _, year, event_type, description = parts
                        country_events.append(
                            {
                                "year": int(year),
                                "event_type": event_type,
                                "description": description,
                            }
                        )

            # Store the events for the current country
            if country_events:
                events[country_name] = country_events

    return events


def show_all_events():
    all_events = load_events(EVENTS_PATH)
    # Print the loaded events for verification
    for country, events in all_events.items():
        print(f"Events for {country}:")
        for event in events:
            print(f"  - {event['year']}: {event['event_type']}")
        print()


def get_name_by_id(input_id):
    """
    Searches for the specified country ID and returns the primary name (name_1).

    Args:
        input_id: The country ID to search for.

    Returns:
        str: The primary name of the country if found, otherwise None.
    """
    try:
        id_int = int(input_id)
        result = countries_df[countries_df["id"] == id_int]
        if not result.empty:
            return result["name_1"].iloc[0]
    except ValueError:
        pass  # Invalid ID format
    return None


def get_fitting_events_for_scdi(scdi_row, events_data):
    """
    Returns event types that fit the given SCDI row.

    Args:
        scdi_row (pd.Series): A row from the SCDIs DataFrame.
        events_data (dict): A dictionary where keys are country names and values are lists of event dictionaries.

    Returns:
        list: A list of event types that fit the SCDI.
    """
    country_a = get_name_by_id(scdi_row["country_a_id"])
    country_b = get_name_by_id(scdi_row["country_b_id"])
    length = scdi_row["length"]
    start_year_a = scdi_row["start_year_a"]
    start_year_b = scdi_row["start_year_b"]

    # Calculate end years for both countries
    end_year_a = start_year_a + length  # -1 # add margin [-2, +2]
    end_year_b = start_year_b + length  # -1 # add margin [-2, +2]

    # Get events for both countries
    country_a_events = events_data.get(str(country_a), [])
    country_b_events = events_data.get(str(country_b), [])

    # Find event types that occurred in the end years for both countries
    fitting_events = set()
    for event in country_a_events:
        if event["year"] == end_year_a:
            for b_event in country_b_events:
                if (
                    b_event["year"] == end_year_b
                    and b_event["event_type"] == event["event_type"]
                ):
                    fitting_events.add(event["event_type"])

    return list(fitting_events)


def browse_all_scdis(print_=False):
    """
    Processes all SCDIs from all country-named parquet files in all indicator-ID-named subfolders.

    Args:
        base_folder (str): The base folder containing indicator-ID-named subfolders.
        events (dict): A dictionary of events where keys are country IDs and values are lists of event dictionaries.
        print_ (bool): If True, prints progress messages. Defaults to False.
    """
    # Get list of indicator folders
    events = load_events(EVENTS_PATH)
    indicator_folders = os.listdir(SCDIS_PATH)

    for x, indicator_folder in enumerate(indicator_folders):
        indicator_path = os.path.join(SCDIS_PATH, indicator_folder)
        if not os.path.isdir(indicator_path):
            continue  # Skip if not a directory

        # Get list of country files in the indicator folder
        country_files = [
            file for file in os.listdir(indicator_path) if file.endswith(".parquet")
        ]

        for y, file in enumerate(country_files):
            country_name = file.replace(".parquet", "")
            file_path = os.path.join(indicator_path, file)

            sdcis_df = pd.read_parquet(file_path)
            total_rows = len(sdcis_df)

            for z, scdi_row in sdcis_df.iterrows():
                indicator_percent = (x + 1) / len(indicator_folders) * 100
                country_percent = (y + 1) / len(country_files) * 100
                row_percent = (z + 1) / total_rows * 100

                clear_output(wait=True)
                print(
                    f"{indicator_folder}: {indicator_percent:.1f}% - {country_name}: {country_percent:.1f}% - 'row_{z}': {row_percent:.1f}%"
                )

                fitting_events = get_fitting_events_for_scdi(scdi_row, events)
                if len(fitting_events) > 0:
                    print(
                        f"Fitting events for SCDI between {scdi_row['country_a_id']} and {scdi_row['country_b_id']}: {fitting_events}"
                    )
                    print(z)
                    return fitting_events

            if print_:
                print(f"{indicator_folder} - {country_name}")

In [ ]:
import pickle

with open(
    r"/workspace/qgi\qgi-4.0\data\recipes\Economic_Growth_recipe.pkl",
    "rb",
) as file:
    loaded_data = pickle.load(file)

show_recipe(loaded_data)

In [ ]:
browse_all_scdis(SCDIS_PATH)

### Original 

In [ ]:
from IPython.display import clear_output
from collections import defaultdict
import pickle
import os
import pandas as pd
import ast

countries_df = pd.read_csv(
    r"/workspace/qgi\qgi-4.0\data\countries.csv"
)  # r"/workspaces/qgi-4.0/data/countries.csv"
EVENTS_PATH = r"/workspace/qgi\qgi-4.0\data\events"  # r"/workspaces/qgi-4.0/data/events"
COUNTRIES_SDCI_PARQUET_PATH = r"/workspace/qgi\qgi-4.0\data\countries_sdci_parquet"  # r"/workspaces/qgi-4.0/data/countries_sdci_parquet"
COUNTRIES_PATTERNS_PATH = r"/workspace/qgi\qgi-4.0\data\country_patterns"  # r"/workspaces/qgi-4.0/data/country_patterns"
RECIPES_PATH = r"/workspace/qgi\qgi-4.0\data\recipes"  # r"/workspaces/qgi-4.0/data/recipes"
COUNTRIES_CLASSES_PKL = r"/workspace/qgi\qgi-4.0\notebooks\countries.pkl"  # r"/workspaces/qgi-4.0/notebooks/countries_classes.pkl"


class Country:
    """
    Represents a country with its associated patterns, SDCIs, and events.

    Attributes:
        name (str): The name of the country.
        id (str): The unique identifier of the country.
        patterns (list[Pattern]): List of patterns associated with this country.
        sdcis (list[SDCI]): List of Single Dyadic Correlating Indexes for this country.
        agg_sdcis (list[SDCI]): List of aggregated SDCIs for this country.
        events (list[dict]): List of events that occurred in this country.
    """

    def __init__(self):
        self.name = ""
        self.id = ""
        self.patterns_count = 0
        self.patterns_path = r""
        self.sdcis_count = 0
        self.sdcis_path = r""
        self.agg_sdcis = 0
        self.agg_sdcis_path = r""
        self.events = list[dict]


class Pattern:
    """
    Represents a pattern between two countries, including various attributes about the pattern.

    Attributes:
        country_a_id (str): ID of the first country in the pattern.
        country_a (str): Name of the first country in the pattern.
        country_b_id (str): ID of the second country in the pattern.
        country_b (str): Name of the second country in the pattern.
        pattern_length (str): Length of the pattern in years.
        start_year_a_fk (str): Start year of the pattern for the first country.
        start_year_b_fk (str): Start year of the pattern for the second country.
        indexes (str): Number of indexes in the pattern.
        avg_corr (str): Average correlation of the pattern.
        orgs (str): Organizations related to the pattern.
        power (str): Power value associated with the pattern.
        sectors (list): Sectors involved in the pattern.
    """

    def __init__(self):
        self.country_a_id = ""
        self.country_a = ""
        self.country_b_id = ""
        self.country_b = ""
        self.pattern_length = ""
        self.start_year_a_fk = ""
        self.start_year_b_fk = ""
        self.indexes = ""
        self.avg_corr = ""
        self.orgs = ""
        self.power = ""
        self.sectors = []


class SDCI:
    """
    Represents a Single Dyadic Correlating Index, capturing the correlation of 2 segments of the same index, but for two countries.

    Attributes:
        unique_id (str): Unique identifier for the SDCI.
        org_fk (str): Organization foreign key.
        main_name_fk (str): Main name foreign key.
        index_name_fk (str): Index name foreign key.
        version_fk (int): Version number.
        country_a_id_fk (int): ID of the first country.
        country_b_id_fk (int): ID of the second country.
        pattern_length_fk (int): Length of the pattern in years.
        correlation (float): Correlation value between the two countries' indexes.
        start_year_a_fk (int): Start year of the pattern for the first country.
        index_values_a (list): Index values for the first country.
        start_year_b_fk (int): Start year of the pattern for the second country.
        index_values_b (list): Index values for the second country.
        creation_date (str): Date when the SDCI was created.
        sector (str): Sector associated with the SDCI.
    """

    def __init__(self):
        self.unique_id = ""
        self.org_fk = ""
        self.main_name_fk = ""
        self.index_name_fk = ""
        self.version_fk = 0
        self.country_a_id_fk = 0
        self.country_b_id_fk = 0
        self.pattern_length_fk = 0
        self.correlation = 0.0
        self.start_year_a_fk = 0
        self.index_values_a = []
        self.start_year_b_fk = 0
        self.index_values_b = []
        self.creation_date = ""
        self.sector = ""


class Recipe:
    """
    Represents a recipe for a specific event type, containing components that describe the indicators and sectors involved.

    Attributes:
        event_type (str): Type of event the recipe is for.
        components (list[tuple]): List of components in the recipe, each being a tuple of (index_name, sector, average_correlation, frequency, frequency_percentage).
    """

    def __init__(self, event_type):
        self.event_type = event_type
        self.components = (
            []
        )  # List of tuples (index_name, sector, average_correlation, frequency, frequency_percentage)

    def add_component(
        self, index_name, sector, average_correlation, frequency, frequency_percentage
    ):
        """
        Adds a component to the recipe.

        Args:
            index_name (str): Name of the index.
            sector (str): Sector associated with the index.
            average_correlation (float): Average correlation value.
            frequency (int): Frequency of the component.
            frequency_percentage (float): Frequency percentage.
        """
        self.components.append(
            (index_name, sector, average_correlation, frequency, frequency_percentage)
        )

    def to_dict(self):
        """
        Converts the recipe to a dictionary.

        Returns:
            dict: Dictionary representation of the recipe.
        """
        return {"event_type": self.event_type, "components": self.components}

    @classmethod
    def from_dict(cls, data):
        """
        Creates a Recipe object from a dictionary.

        Args:
            data (dict): Dictionary containing recipe data.

        Returns:
            Recipe: Recipe object created from the dictionary.
        """
        recipe = cls(data["event_type"])
        for component in data["components"]:
            recipe.add_component(*component)
        return recipe

In [ ]:
## TODO: Can the classes above be upgraded/professionalized? making them more robust?
## TODO: Backtesting recpies on all patterns or backtesting all patterns on all recipes?
## TODO: When benchmarking, don't we need index levels? just index name, sector and correlation are enough?
## TODO: We can't keep all this code in a notebook, so how would a pipeline that utilizes
## these functions and gives us the right results look like? what results?
## TODO: Early Warning System

In [ ]:
## General Utils
def save_class_as_pkl(class_var, path_and_name: str):
    """
    Saves a class instance to a pickle file.

    Args:
        class_var: The class instance to be saved.
        path_and_name (str): The full path and filename where the pickle file will be saved.
    """
    with open(path_and_name, "wb") as f:
        pickle.dump(class_var, f)


def read_pkl_as_class(pkl_path: str):
    """
    Reads a pickle file and returns the class instance.

    Args:
        pkl_path (str): The path to the pickle file.

    Returns:
        The class instance stored in the pickle file.
    """
    with open(pkl_path, "rb") as f:
        return pickle.load(f)


def count_event_types(country_classes: list[Country]):
    """
    Counts the occurrences of each event type across all countries.

    Args:
        country_classes (list[Country]): List of Country instances.

    Returns:
        dict: A dictionary where keys are event types and values are their counts.
    """
    event_counts = defaultdict(int)
    for country in country_classes:
        for event in country.events:
            event_type = event["event_type"]
            event_counts[event_type] += 1
    return dict(event_counts)


def rank_event_types(country_classes):
    """
    Ranks event types by their frequency across all countries.

    Args:
        country_classes: List of Country instances.

    Returns:
        list: A list of tuples sorted by event frequency in descending order, where each tuple contains an event type and its count.
    """
    event_counts = count_event_types(country_classes)
    sorted_events = sorted(event_counts.items(), key=lambda x: x[1], reverse=True)
    return sorted_events


def update_and_get_countries_events(
    countries_pkl_file_name: str = COUNTRIES_CLASSES_PKL,
):
    """
    Updates country classes with the latest events data and returns the updated list.

    This function loads country classes from a pickle file, updates each country's events data,
    saves the updated classes back to the pickle file, and returns the updated list of country classes.

    Args:
        countries_pkl_file_name (str): The filename of the pickle file containing country classes. Defaults to COUNTRIES_CLASSES_PKL.

    Returns:
        list[Country]: The updated list of Country instances with fresh events data.
    """
    country_classes = read_pkl_as_class(countries_pkl_file_name)
    for country in country_classes:
        country.events = read_events_data(country.name)
    save_class_as_pkl(country_classes, countries_pkl_file_name)

    # returning country classes
    return country_classes


def make_pattern(row):
    """
    Creates a Pattern instance from a DataFrame row.

    Args:
        row: A DataFrame row containing pattern data.

    Returns:
        Pattern: A Pattern instance populated with data from the row.
    """
    pattern = Pattern()
    pattern.country_a_id = int(row["country_a_id_fk"])
    pattern.country_a = get_name_by_id(pattern.country_a_id)
    pattern.country_b_id = int(row["country_b_id_fk"])
    pattern.country_b = get_name_by_id(pattern.country_b_id)
    pattern.pattern_length = int(row["pattern_length_fk"])
    pattern.start_year_a_fk = int(row["start_year_a_fk"])
    pattern.start_year_b_fk = int(row["start_year_b_fk"])
    pattern.indexes = int(row["indexes"])
    pattern.avg_corr = round(row["avg_corr"], 6)
    try:
        pattern.orgs = int(row["orgs"])
        pattern.sectors = ast.literal_eval(row["Sectors"])
        pattern.power = round(row["Power"], 6)
    except KeyError:
        pattern.orgs = ""
        pattern.sectors = ""
        pattern.power = ""
        # print("Erring row: ")
        # print(row)
        # print("------------")

    return pattern


def get_name_by_id(input_id):
    """
    Searches for the specified country ID and returns the primary name (name_1).

    Args:
        input_id: The country ID to search for.

    Returns:
        str: The primary name of the country if found, otherwise None.
    """
    try:
        id_int = int(input_id)
        result = countries_df[countries_df["id"] == id_int]
        if not result.empty:
            return result["name_1"].iloc[0]
    except ValueError:
        pass  # Invalid ID format
    return None


def load_actual_events():
    """
    Loads actual historical events for all countries from the EVENTS_PATH.

    Returns:
        dict: A dictionary where keys are country IDs and values are lists of events
    """
    actual_events = {}
    country_classes = update_and_get_countries_events()

    for country in country_classes:
        country_events = read_events_data(country.name)
        actual_events[country.id] = country_events

    return actual_events


def sort_patterns_by_power(patterns_list):
    """
    Sorts patterns by their power attribute in descending order.

    Patterns with empty power strings are placed at the end of the list.

    Args:
        patterns_list: List of Pattern instances

    Returns:
        List of Pattern instances sorted by power (highest first)
    """

    def power_key(pattern):
        power = str(pattern.power)
        if power.strip() == "":
            return float("-inf")
        try:
            return float(power)
        except ValueError:
            return float("-inf")

    return sorted(patterns_list, key=power_key, reverse=True)


## Recipe functions
def read_events_data(country_name, show_errors=False):
    """
    Reads and parses event data for a specific country from a text file.

    Args:
        country_name (str): The name of the country whose events to read.
        show_errors (bool): If True, prints error messages for missing files. Defaults to False.

    Returns:
        list[dict]: A list of dictionaries, where each dictionary contains event details:
            - "year" (int): The year the event occurred
            - "event_type" (str): The type of event
            - "description" (str): A description of the event

    Raises:
        Exception: If there's an error reading the file (and show_errors is True)
    """
    try:
        path = os.path.join(EVENTS_PATH, f"{country_name}.txt")
        with open(path, "r") as f:
            events = f.readlines()
        parsed_events = []
        for event in events:
            parts = event.strip().split(" | ")
            if len(parts) == 4:
                _, year, event_type, description = parts
                parsed_events.append(
                    {
                        "year": int(year),
                        "event_type": event_type,
                        "description": description,
                    }
                )
        return parsed_events
    except Exception as e:  # FileNotFoundError:
        if show_errors:
            if "No such file or directory" not in str(e):
                print(str(e))
                print(path)
        return []


def get_patterns_leading_to_event(event_type: str, margin: int = 0) -> list[Pattern]:
    """
    Identifies patterns that lead to a specific event type by checking if both countries
    in the pattern have the event occur within a specified margin of years after
    the pattern's end year.

    Args:
        event_type (str): The type of event to identify patterns for
        margin (int): The number of years before/after the pattern's end year to
            consider for events. Defaults to 0.

    Returns:
        list[Pattern]: A list of Pattern instances that lead to the specified event type
    """
    event_patterns = []
    country_classes: list[Country] = update_and_get_countries_events()

    for country in country_classes:
        # Get event years for this country
        event_years = get_event_years(country, event_type)

        # Check each pattern for this country
        for pattern in country.patterns:
            # Calculate pattern end years
            end_year_a = pattern.start_year_a_fk + pattern.pattern_length
            end_year_b = pattern.start_year_b_fk + pattern.pattern_length

            # Check if country A has an event within the margin
            has_event_a = check_event_within_margin(end_year_a, event_years, margin)

            # Get country B
            country_b = find_country_by_id(country_classes, pattern.country_b_id)
            if country_b:
                country_b_event_years = get_event_years(country_b, event_type)
                has_event_b = check_event_within_margin(
                    end_year_b, country_b_event_years, margin
                )

                if has_event_a and has_event_b:
                    event_patterns.append(pattern)

    return event_patterns


def get_event_years(country: Country, event_type: str) -> list[int]:
    """
    Extracts years when a specific event type occurred for a given country.

    Args:
        country (Country): The Country instance to check
        event_type (str): The type of event to look for

    Returns:
        list[int]: A list of years when the event occurred
    """
    event_years = []
    for event in country.events:
        if event["event_type"] == event_type:
            event_years.append(event["year"])
    return event_years


def check_event_within_margin(
    end_year: int, event_years: list[int], margin: int
) -> bool:
    """
    Checks if any event year falls within the specified margin of the pattern's end year.

    Args:
        end_year (int): The end year of the pattern
        event_years (list[int]): List of years when the event occurred
        margin (int): The number of years to consider on either side of end_year

    Returns:
        bool: True if at least one event year is within the margin, False otherwise
    """
    for year in event_years:
        if abs(end_year - year) <= margin:
            return True
    return False


def find_country_by_id(countries: list[Country], country_id: str) -> Country | None:
    """
    Finds a country in a list based on its ID.

    Args:
        countries (list[Country]): List of Country instances to search
        country_id (str): The ID of the country to find

    Returns:
        Country | None: The Country instance if found, None otherwise
    """
    for country in countries:
        if country.id == country_id:
            return country
    return None


def show_recipe(recipe: Recipe, print_out=True):
    """
    Displays a recipe's details in a readable format.

    Args:
        recipe (Recipe): The Recipe instance to display
        print_out (bool): If True, prints the recipe details. If False, returns them as a dictionary. Defaults to True.

    Returns:
        dict: Recipe details as a dictionary if print_out is False
    """
    if print_out:
        print(f"Event Type: {recipe.event_type}")
        print(f"Number of Components: {len(recipe.components)}")
        print("-" * 50)
        for component in recipe.components:
            index_name, sector, avg_corr, freq, freq_percent = component
            print(f"Index: {index_name}")
            print(f"Sector: {sector}")
            print(f"Average Correlation: {avg_corr:.2f}")
            print(f"Frequency: {freq} ({freq_percent:.1f}%)")
            print("-" * 50)

    return {
        "event_type": recipe.event_type,
        "components": [
            {
                "index_name": component[0],
                "sector": component[1],
                "average_correlation": component[2],
                "frequency": component[3],
                "frequency_percentage": component[4],
            }
            for component in recipe.components
        ],
    }


def show_country(country: Country, print_out=True):
    """
    Displays country information in a readable format.

    Args:
        country (Country): The Country instance to display.
        print_out (bool): If True, prints the country information. If False, returns the information as a dictionary.

    Returns:
        dict: Country information as a dictionary if print_out is False.
    """
    if print_out:
        print(f"Country Name: {country.name}")
        print(f"Country ID: {country.id}")
        print(f"Patterns Count: {country.patterns_count}")
        print(f"Patterns Path: {country.patterns_path}")
        print(f"SDCIs Count: {country.sdcis_count}")
        print(f"SDCIs Path: {country.sdcis_path}")
        print(f"Aggregated SDCIs: {country.agg_sdcis}")
        print(f"Aggregated SDCIs Path: {country.agg_sdcis_path}")
        print(f"Number of Events: {len(country.events)}")
    return {
        "name": country.name,
        "id": country.id,
        "patterns_count": country.patterns_count,
        "patterns_path": country.patterns_path,
        "sdcis_count": country.sdcis_count,
        "sdcis_path": country.sdcis_path,
        "agg_sdcis": country.agg_sdcis,
        "agg_sdcis_path": country.agg_sdcis_path,
        "events_count": len(country.events),
    }


def create_event_recipe(
    sdcis_df: pd.DataFrame, event_type: str, margin=0, take_common_sdcis=False
) -> Recipe:
    """
    Creates a comprehensive recipe for a specific event type based on patterns leading to that event.

    This function identifies patterns that lead to the specified event type and constructs a recipe
    containing the most relevant SDCIs (Single Dyadic Correlating Indexes) associated with those patterns.

    Args:
        sdcis_df (pd.DataFrame): DataFrame containing all SDCIs.
        event_type (str): The type of event for which to create the recipe.
        margin (int): Number of years before/after the pattern's end year to consider for events. Defaults to 0.
        take_common_sdcis (bool): If True, only include SDCIs common to all patterns; if False, include all SDCIs. Defaults to False.

    Returns:
        Recipe: A Recipe object containing components that describe the recipe.

    Raises:
        Prints a message if no patterns are found for the event type or if no common SDCIs are found when take_common_sdcis is True.
    """
    recipe = Recipe(event_type)
    component_counts = defaultdict(int)
    correlation_sums = defaultdict(float)
    patterns = get_patterns_leading_to_event(event_type, margin)

    if not patterns:
        print(f"No patterns found for event type '{event_type}' with margin {margin}.")
        return recipe

    if take_common_sdcis:
        # Find SDCIs common to all patterns
        common_sdcis = None
        for pattern in patterns:
            sdcis = get_sdcis_for_pattern(pattern, sdcis_df)
            current_sdcis = set(sdcis[["index_name_fk", "sector"]].apply(tuple, axis=1))
            if common_sdcis is None:
                common_sdcis = current_sdcis
            else:
                common_sdcis &= current_sdcis
            if not common_sdcis:
                break  # No need to continue if already empty

        if not common_sdcis:
            print(f"No common SDCIs found for event type '{event_type}'.")
            return recipe

        # Collect only common SDCIs
        for pattern in patterns:
            sdcis = get_sdcis_for_pattern(pattern, sdcis_df)
            for _, sdci in sdcis.iterrows():
                key = (sdci["index_name_fk"], sdci["sector"])
                if key in common_sdcis:
                    component_counts[key] += 1
                    correlation_sums[key] += sdci["correlation"]
    else:
        # Collect all SDCIs from all patterns
        for pattern in patterns:
            sdcis = get_sdcis_for_pattern(pattern, sdcis_df)
            for _, sdci in sdcis.iterrows():
                key = (sdci["index_name_fk"], sdci["sector"])
                component_counts[key] += 1
                correlation_sums[key] += sdci["correlation"]

    # Calculate statistics for each SDCI and build the recipe
    for key in component_counts:
        index_name, sector = key
        frequency = component_counts[key]
        avg_corr = correlation_sums[key] / frequency
        freq_percent = (frequency / len(patterns)) * 100

        recipe.add_component(
            index_name=index_name,
            sector=sector,
            average_correlation=avg_corr,
            frequency=frequency,
            frequency_percentage=freq_percent,
        )

    # Sort components by frequency descending
    recipe.components.sort(key=lambda x: x[3], reverse=True)

    return recipe


def load_sdcis_from_parquet(folder_path):
    """
    Loads SDCIs from Parquet files in a specified folder.

    Args:
        folder_path (str): Path to the folder containing Parquet files.

    Returns:
        pd.DataFrame: A concatenated DataFrame of all SDCIs from the Parquet files.
    """
    sdcis = []
    for file in os.listdir(folder_path):
        if file.endswith(".parquet"):
            file_path = os.path.join(folder_path, file)
            sdcis.append(pd.read_parquet(file_path))
    return pd.concat(sdcis, ignore_index=True)


def get_sdcis_for_pattern(pattern: Pattern, sdcis_df):
    """
    Retrieves SDCIs for a specific pattern from a DataFrame.

    Args:
        pattern (Pattern): The pattern for which to retrieve SDCIs.
        sdcis_df (pd.DataFrame): DataFrame containing all SDCIs.

    Returns:
        pd.DataFrame: A filtered DataFrame containing SDCIs that match the pattern's attributes.
    """
    return sdcis_df.query(
        f"country_a_id_fk == {pattern.country_a_id} and "
        f"country_b_id_fk == {pattern.country_b_id} and "
        f"pattern_length_fk == {pattern.pattern_length} and "
        f"start_year_a_fk == {pattern.start_year_a_fk} and "
        f"start_year_b_fk == {pattern.start_year_b_fk}"
    )


def read_all_recipes(recipes_folder):
    """
    Reads all recipes from a folder containing pickled recipe files.

    Args:
        recipes_folder (str): Path to the folder containing recipe files.

    Returns:
        dict: A dictionary where keys are event types and values are the corresponding Recipe objects.
    """
    recipes = {}
    for filename in os.listdir(recipes_folder):
        if filename.endswith("_recipe.pkl"):
            event_name = filename.replace("_recipe.pkl", "")
            recipe_path = os.path.join(recipes_folder, filename)
            recipe = read_pkl_as_class(recipe_path)
            recipes[event_name] = recipe
    return recipes

In [ ]:
## Loading SDCIs parquet and countries class pkl
sdcis_df = load_sdcis_from_parquet(
    COUNTRIES_SDCI_PARQUET_PATH
)  ## TODO: replace with querying parquet right away
country_classes = update_and_get_countries_events()
event_frequencies = rank_event_types(country_classes)
all_recipes = read_all_recipes(RECIPES_PATH)
actual_events = load_actual_events()

In [ ]:
all_recipes = read_all_recipes(RECIPES_PATH)
for rec in list(all_recipes.keys()):
    if "Economic_Growth" == rec:
        show_recipe(all_recipes[rec])

In [ ]:
def benchmark_pattern(pattern: Pattern, recipe: Recipe):
    """
    Benchmarks a pattern against a recipe to determine how similar they are.

    This function calculates a similarity score based on how many components the pattern shares with the recipe
    and their respective correlations.

    Args:
        pattern (Pattern): The pattern to benchmark.
        recipe (Recipe): The recipe to benchmark against.

    Returns:
        float: The similarity score between the pattern and the recipe.
    """
    sdcis = get_sdcis_for_pattern(pattern, sdcis_df)
    pattern_components = defaultdict(list)

    for _, sdci in sdcis.iterrows():
        key = (sdci["index_name_fk"], sdci["sector"])
        pattern_components[key].append(sdci["correlation"])

    # Calculate similarity score
    similarity_score = 0
    for component in recipe.components:
        index_name, sector, avg_corr, freq, _ = component
        if (index_name, sector) in pattern_components:
            pattern_corr = sum(pattern_components[(index_name, sector)]) / len(
                pattern_components[(index_name, sector)]
            )
            similarity_score += pattern_corr * freq  # Weighted by frequency in recipe

    return similarity_score


def benchmark_pattern_against_all_recipes(pattern, all_recipes):
    """
    Benchmarks a pattern against all available recipes with progress tracking.

    This function calculates similarity scores between a pattern and all recipes,
    displays progress percentage, and returns the results sorted by similarity score.

    Args:
        pattern: The pattern to benchmark.
        all_recipes: A dictionary of all available recipes.

    Returns:
        list: A list of tuples containing event names and their corresponding similarity scores, sorted by score.
    """
    results = []
    total_recipes = len(all_recipes)
    for i, (event_name, recipe) in enumerate(all_recipes.items()):
        similarity_score = benchmark_pattern(pattern, recipe)
        results.append((event_name, similarity_score))

        # Calculate and display progress percentage
        progress = (i + 1) / total_recipes * 100
        clear_output(wait=True)
        print(f"{progress:.1f}%")

    results.sort(key=lambda x: x[1], reverse=True)
    return results

### Backtesting

#### Backtesting Patterns Against Recipes

This approach checks how well your historical patterns would have predicted events using your recipes. It is better for assessing prediction accuracy at the pattern level.

How It Works:

1. Load Data:
    - Patterns: Historical patterns from your data
    - Recipes: Predefined event recipes
    - Actual Events: Historical record of events that occurred
2. Benchmark Each Pattern:
    - For each pattern, calculate similarity scores against all recipes
    - The highest similarity score determines the predicted event
3. Compare Predictions to Actuals:
    - Check if the predicted event matches what actually happened
    - Record the result for performance metrics

Let's say we have:

- Patterns: 100 historical patterns from various country pairs
- Recipes: 10 event recipes (like "Military Conflict", "Economic Growth")
- Actual Events: Historical record of events that occurred after these patterns

For each of the 100 patterns:

- Calculate similarity scores against all 10 recipes
- The recipe with the highest score is the predicted event
- Compare this prediction to what actually happened

Why This Approach?

- Validation: It validates if your recipes actually work for predicting events
- Improvement: Identifies which recipes perform well and which need refinement
- Performance Metrics: Provides accuracy, precision, and recall to quantify system effectiveness

In [ ]:
def get_actual_event_for_pattern(pattern, actual_events):
    """
    Finds the actual event that occurred after the pattern.

    Args:
        pattern (Pattern): The pattern to check
        actual_events (dict): A dictionary of events with their occurrence years

    Returns:
        str: The actual event type that occurred, or "No Event" if none found
    """
    # Example implementation logic:
    # 1. Calculate the end year of the pattern
    end_year = pattern.start_year_a_fk + pattern.pattern_length

    # 2. Check if any event occurred within a reasonable timeframe after the pattern
    country_a_events = actual_events.get(pattern.country_a_id, [])
    country_b_events = actual_events.get(pattern.country_b_id, [])

    # 3. Implement your logic to determine if an event occurred after the pattern
    # This is a placeholder for your actual implementation
    for event in country_a_events + country_b_events:
        if event["year"] > end_year and event["year"] <= end_year + 5:
            return event["event_type"]

    return "No Event"


def calculate_accuracy(results):
    correct = sum(
        1 for result in results if result["predicted_event"] == result["actual_event"]
    )
    total = len(results)
    return correct / total if total > 0 else 0


def calculate_precision_recall(results):
    """
    Calculate precision and recall for the backtesting results.

    Args:
        results (list): List of backtesting results

    Returns:
        tuple: (precision, recall)
    """
    tp = sum(
        1
        for r in results
        if r["predicted_event"] == r["actual_event"] and r["actual_event"] != "No Event"
    )
    fp = sum(
        1
        for r in results
        if r["predicted_event"] != r["actual_event"] and r["actual_event"] == "No Event"
    )
    fn = sum(
        1
        for r in results
        if r["predicted_event"] == "No Prediction" and r["actual_event"] != "No Event"
    )

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return precision, recall


def test_single_pattern(pattern, all_recipes):
    """Test a single pattern against all recipes with progress tracking."""
    similarity_scores = benchmark_pattern_against_all_recipes(pattern, all_recipes)
    predicted_event = similarity_scores[0][0] if similarity_scores else "No Prediction"
    actual_event = get_actual_event_for_pattern(pattern, actual_events)

    return {
        "pattern": pattern,
        "predicted_event": predicted_event,
        "actual_event": actual_event,
        "similarity_score": similarity_scores[0][1] if similarity_scores else 0,
    }


def backtest_and_report_progress(
    patterns, all_recipes, checkpoint_file="backtest_checkpoint.pkl"
):
    """
    Backtests patterns against recipes with progress reporting and returns successful patterns.

    Args:
        patterns: List of patterns to backtest
        all_recipes: Dictionary of all recipes
        checkpoint_file: File path to save/load checkpoint data

    Returns:
        List of patterns that were successfully predicted
    """
    # Load checkpoint if it exists
    if os.path.exists(checkpoint_file):
        checkpoint = read_pkl_as_class(checkpoint_file)
        processed_patterns = checkpoint.get("processed_patterns", 0)
        all_results = checkpoint.get("results", [])
    else:
        processed_patterns = 0
        all_results = []

    total_patterns = len(patterns)
    remaining_patterns = total_patterns - processed_patterns

    if remaining_patterns <= 0:
        print("Backtesting already completed.")
        return [
            result["pattern"]
            for result in all_results
            if result["predicted_event"] == result["actual_event"]
        ]

    # Process remaining patterns
    for i in range(processed_patterns, total_patterns):
        pattern = patterns[i]
        similarity_scores = benchmark_pattern_against_all_recipes(pattern, all_recipes)
        predicted_event = (
            similarity_scores[0][0] if similarity_scores else "No Prediction"
        )
        actual_event = get_actual_event_for_pattern(pattern, actual_events)

        all_results.append(
            {
                "pattern": pattern,
                "predicted_event": predicted_event,
                "actual_event": actual_event,
                "similarity_score": similarity_scores[0][1] if similarity_scores else 0,
            }
        )

        # # Update checkpoint
        # checkpoint_data = {
        #     "processed_patterns": i + 1,
        #     "results": all_results
        # }
        # save_class_as_pkl(checkpoint_data, checkpoint_file)

        # Calculate and display progress percentage
        progress = (i + 1) / total_patterns * 100
        clear_output(wait=True)
        print(f"Progress: {progress:.1f}%")

    # Filter successful predictions
    successful_patterns = [
        result["pattern"]
        for result in all_results
        if result["predicted_event"] == result["actual_event"]
    ]

    return successful_patterns

##### Backtesting first 100 patterns, ranked by power

In [ ]:
successful_patterns = backtest_and_report_progress(
    sort_patterns_by_power(patterns_list)[:100], all_recipes
)

##### Backtesting a single pattern

In [ ]:
# Test with a single pattern
single_pattern = patterns_list[2]
result = test_single_pattern(single_pattern, all_recipes)
print(f"Pattern: {result['pattern'].country_a} - {result['pattern'].country_b}")
print(f"Predicted: {result['predicted_event']}")
print(f"Actual: {result['actual_event']}")
print(f"Score: {result['similarity_score']:.2f}")

#### Backtesting Recipes Against Patterns

This approach checks how well your recipes would have identified patterns that actually led to events. It is better for evaluating how well your recipes capture the essence of specific events.

In [ ]:
def backtest_recipes_against_patterns(event_type, recipe, historical_patterns):
    # Filter patterns that actually led to the event
    relevant_patterns = [
        p for p in historical_patterns if event_type in get_events_for_pattern(p)
    ]

    # Benchmark each relevant pattern against the recipe
    scores = [benchmark_pattern(p, recipe) for p in relevant_patterns]

    # Analyze distribution of scores
    threshold = determine_threshold(scores)
    detection_rate = sum(1 for score in scores if score >= threshold) / len(scores)

    return detection_rate, scores


def get_events_for_pattern(pattern):
    # Implement logic to retrieve events that occurred after this pattern
    pass


def determine_threshold(scores):
    # Implement logic to determine an appropriate threshold for detection
    pass

### Pattern-Recipe Similarity Benchmarking

In [ ]:
df = pd.read_csv(r"/workspace/qgi\qgi-4.0\data\ppr.csv")
pattern = make_pattern(df.sort_values(by = ["indexes"], ascending = False).iloc[0, :])

# economic_growth_recipe = read_pkl_as_class(os.path.join(RECIPES_PATH, r"\Economic_Growth_recipe.pkl"))
# similarity_score = benchmark_pattern(pattern, economic_growth_recipe)
# print(f"Similarity Score: {similarity_score}")

results = benchmark_pattern_against_all_recipes(pattern, all_recipes)
for event_name, score in results:
    print(f"Event: {event_name.replace("_", " ")}, Similarity Score: {score:.2f}")

### Generate and save all recipes

In [ ]:
for event_type, _ in event_frequencies:
    event_recipe = create_event_recipe(sdcis_df, event_type)
    if event_recipe.components:
        save_class_as_pkl(
            event_recipe,
            os.path.join(RECIPES_PATH, f"{event_type.replace(' ', '_')}_recipe.pkl"),
        )
    else:
        print(f"No components for recipe '{event_type}', skipping save.")

### Get SDCIs Recipe for Events

In [ ]:
EVENT = "Economic Growth"
event_recipe = create_event_recipe(sdcis_df, EVENT, take_common_sdcis=True)

for component in event_recipe.components:
    index_name, sector, avg_corr, freq, freq_percent = component
    print(f"Index: {index_name}, Sector: {sector}")
    print(f"Average Correlation: {avg_corr:.2f}")
    print(f"Frequency: {freq} ({freq_percent:.1f}%)")
    print("-" * 50)

In [ ]:
EVENT = "Economic Growth"
event_recipe = create_event_recipe(sdcis_df, EVENT, take_common_sdcis=False)

for component in event_recipe.components:
    index_name, sector, avg_corr, freq, freq_percent = component
    print(f"Index: {index_name}, Sector: {sector}")
    print(f"Average Correlation: {avg_corr:.2f}")
    print(f"Frequency: {freq} ({freq_percent:.1f}%)")
    print("-" * 50)

### Get Events Frequency

In [ ]:
for event_type, count in event_frequencies:
    print(f"{event_type}: {count}")